# 11 — Lightweight Mel CNN Baseline

Optional PyTorch experiment. Uses patient-disjoint train/validation/test splits; class weights are fit from training labels, early stopping and LR reduction use validation loss, and only training cycles receive noise/time-shift augmentation. If the optional framework is unavailable, install only the optional requirements file.

In [ ]:
import sys
from pathlib import Path
ROOT=Path.cwd().resolve()
if ROOT.name=='post_05': ROOT=ROOT.parent.parent
elif ROOT.name=='notebooks': ROOT=ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
import json, pandas as pd
from src.post_05.common import FEATURES_DIR, MODELS_DIR, METRICS_DIR, PREDICTIONS_DIR, FIGURES_DIR, ensure_output_dirs, new_artifact_path
from src.post_05.cnn_utils import fit_cnn
from src.post_05.patient_level import classification_metrics
ensure_output_dirs()
try:
    import torch
except ImportError as exc:
    raise RuntimeError('PyTorch is optional and is not installed. Install requirements-post-05-optional.txt to run CNN experiments.') from exc
data=pd.read_csv(FEATURES_DIR/'cycle_features.csv').merge(pd.read_csv(FEATURES_DIR/'patient_splits.csv')[['patient_id','split']],on='patient_id',validate='many_to_one')
train=data[data.split.eq('train')]; val=data[data.split.eq('validation')]; test=data[data.split.eq('test')]
assert not (set(train.patient_id)&set(val.patient_id)|set(train.patient_id)&set(test.patient_id)|set(val.patient_id)&set(test.patient_id))
pred,info=fit_cnn(train,val,test,MODELS_DIR/'cnn_mel_baseline.pt',channels=1)
labels=sorted(set(test.diagnosis.astype(str))|set(pred.prediction.astype(str)))
metrics=classification_metrics(test.diagnosis.astype(str),pred.prediction.astype(str),labels)
new_artifact_path(PREDICTIONS_DIR/'cnn_mel_baseline_cycle_predictions.csv'); pred.to_csv(PREDICTIONS_DIR/'cnn_mel_baseline_cycle_predictions.csv',index=False)
new_artifact_path(METRICS_DIR/'cnn_mel_baseline.json').write_text(json.dumps({'metrics':metrics,'training':info},indent=2))
display(pd.DataFrame(info['history']))
print(metrics)